**Multimodal Sports Contact Analysis — Data Exploration Pipeline**  
**Presented by: Reza Saadatyar (2026)**  
E-mail: R.Saadatyar90@gmail.com

---

### Notebook 01: Data Exploration

This notebook is the first step of the project. The goal is to understand the raw player-tracking and contact-label data **before** feature engineering or model training.

The notebook answers five basic questions:

1. What files and variables are available?
2. How large are the tracking and label tables?
3. Are there missing values or obvious data-quality issues?
4. How imbalanced is the contact target?
5. How are labelled contact events connected to player movement over time?

## 1. Imported Libraries

We start with general Python libraries, plotting tools, and the project utilities already defined in `src/`.

In [ ]:
# ================================== Imported Libraries ============================================

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ----------------------------------- 1. Find Repository Root --------------------------------------
def find_repository_root(start_path=None):
    """
    Walk upward from the current directory until the project README and code
    directory are found.

    This makes the notebook work whether it is launched from the repository
    root or directly from code/python/notebooks.
    """
    start = Path(start_path or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate

    raise RuntimeError(
        "Repository root could not be located. "
        "Open this notebook from inside the cloned project repository."
    )


ROOT_DIR = find_repository_root()
PYTHON_DIR = ROOT_DIR / "code" / "python"

if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

# ---------------------------------- 2. Project Utility Imports ------------------------------------
from src.utils.data_loader import load_competition_tables
from src.utils.hardware_verification import verify_hardware
from src.utils.reproducibility import set_seed

print(f"✅ Repository root: {ROOT_DIR}")
print(f"✅ Python source path: {PYTHON_DIR}")

## 2. Reproducibility and Hardware

A fixed random seed makes sampling and later model experiments reproducible.  
The hardware check tells us whether PyTorch can use a CUDA GPU.

In [ ]:
# =============================== Reproducibility & Hardware =====================================

SEED = 42
set_seed(SEED, deterministic=True)

device = verify_hardware()
print(f"\nGlobal computation device set to: {device}")

## 3. Project Directories

Raw Kaggle files are expected inside `data/raw/`.

Generated tables and figures are written to local output folders, which are excluded from GitHub by `.gitignore`.

In [ ]:
# ================================ Project Directory Setup =======================================

DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
OUTPUTS_DIR = ROOT_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
REPORTS_DIR = OUTPUTS_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TRACKING_FILE = DATA_DIR / "train_player_tracking.csv"
LABEL_FILE = DATA_DIR / "train_labels.csv"

print(f"Raw data directory : {DATA_DIR}")
print(f"Tracking file      : {TRACKING_FILE}")
print(f"Label file         : {LABEL_FILE}")

## 4. Check Whether the Dataset Is Available

The raw competition data are not stored in this repository.

If the two required CSV files are missing, follow `data/README.md` to download the Kaggle competition data first.

In [ ]:
# ===================================== Data Availability =========================================

required_files = {
    "Player tracking": TRACKING_FILE,
    "Contact labels": LABEL_FILE,
}

missing_files = []

for file_name, file_path in required_files.items():
    status = "✅ FOUND" if file_path.exists() else "❌ MISSING"
    print(f"{status:10s} | {file_name:20s} | {file_path}")

    if not file_path.exists():
        missing_files.append(file_path)

DATA_AVAILABLE = len(missing_files) == 0

if not DATA_AVAILABLE:
    print("\nThe raw dataset is not available yet.")
    print("Open data/README.md and complete the Kaggle download step before continuing.")
else:
    print("\n✅ Required CSV files are available. The exploration can continue.")

## 5. Load Tracking and Contact Labels

The loader checks the core schema before returning the DataFrames.  
This prevents the analysis from silently continuing with unexpected columns.

In [ ]:
# ===================================== Load Dataset ===============================================

if DATA_AVAILABLE:
    tracking, labels = load_competition_tables(DATA_DIR)

    print("\nDataset loaded successfully.")
    print(f"Tracking shape: {tracking.shape}")
    print(f"Labels shape  : {labels.shape}")
else:
    tracking = None
    labels = None
    print("⏸️ Dataset loading skipped because the raw CSV files are missing.")

## 6. First Look at the Tables

Before any modelling, inspect column names, data types, and a few example rows.

In [ ]:
# ================================== Tracking Table Overview =======================================

if DATA_AVAILABLE:
    print("TRACKING COLUMNS")
    print("-" * 100)
    print(tracking.columns.tolist())

    display(tracking.head())

    print("\nTRACKING DATA TYPES")
    print("-" * 100)
    display(
        tracking.dtypes
        .astype(str)
        .rename("dtype")
        .to_frame()
    )

In [ ]:
# =================================== Label Table Overview =========================================

if DATA_AVAILABLE:
    print("LABEL COLUMNS")
    print("-" * 100)
    print(labels.columns.tolist())

    display(labels.head())

    print("\nLABEL DATA TYPES")
    print("-" * 100)
    display(
        labels.dtypes
        .astype(str)
        .rename("dtype")
        .to_frame()
    )

## 7. Basic Dataset Size

Here we count rows, unique plays, unique players, and labelled positive contacts.

These values provide a simple audit record for the dataset version used in the project.

In [ ]:
# ================================= Basic Dataset Statistics =======================================

if DATA_AVAILABLE:
    tracking_summary = {
        "tracking_rows": len(tracking),
        "tracking_columns": tracking.shape[1],
        "unique_game_plays_tracking": tracking["game_play"].nunique(),
        "unique_players_tracking": tracking["nfl_player_id"].nunique(),
        "label_rows": len(labels),
        "label_columns": labels.shape[1],
        "unique_game_plays_labels": labels["game_play"].nunique(),
        "positive_contacts": int(labels["contact"].sum()),
        "negative_contacts": int((labels["contact"] == 0).sum()),
    }

    summary_df = pd.DataFrame(
        tracking_summary.items(),
        columns=["metric", "value"],
    )

    display(summary_df)

    summary_path = REPORTS_DIR / "01_dataset_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"✅ Summary saved to: {summary_path}")

## 8. Missing Values

Missingness is checked separately for tracking and labels.

A variable with missing values is not automatically unusable. The important step is to understand **where**, **how often**, and **why** values are missing before choosing an imputation or exclusion strategy.

In [ ]:
# ==================================== Missing-Value Audit ==========================================

def missing_value_report(df):
    """
    Return missing-value count and percentage for each column.
    """
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100,
    })

    report = report.sort_values(
        "missing_percent",
        ascending=False,
    )

    return report


if DATA_AVAILABLE:
    tracking_missing = missing_value_report(tracking)
    label_missing = missing_value_report(labels)

    print("TRACKING MISSINGNESS")
    display(tracking_missing)

    print("\nLABEL MISSINGNESS")
    display(label_missing)

## 9. Duplicate Checks

Exact duplicated rows can indicate repeated records or file-processing issues.

The `contact_id` field should also be inspected because it identifies labelled contact candidates.

In [ ]:
# ====================================== Duplicate Audit ===========================================

if DATA_AVAILABLE:
    duplicate_summary = pd.DataFrame(
        {
            "check": [
                "Exact duplicate tracking rows",
                "Exact duplicate label rows",
                "Duplicate contact_id values",
            ],
            "count": [
                int(tracking.duplicated().sum()),
                int(labels.duplicated().sum()),
                int(labels["contact_id"].duplicated().sum()),
            ],
        }
    )

    display(duplicate_summary)

## 10. Class Imbalance

The target variable is `contact`:

- `1` = contact event
- `0` = non-contact candidate

Because positive contacts are relatively uncommon, later modelling should not rely on accuracy alone.

In [ ]:
# ======================================= Class Balance =============================================

if DATA_AVAILABLE:
    class_counts = (
        labels["contact"]
        .value_counts()
        .sort_index()
        .rename_axis("contact")
        .reset_index(name="count")
    )

    class_counts["percentage"] = (
        class_counts["count"] / class_counts["count"].sum() * 100
    )

    display(class_counts)

    positive_rate = labels["contact"].mean() * 100
    print(f"Positive contact rate: {positive_rate:.4f}%")

    ax = class_counts.plot(
        x="contact",
        y="count",
        kind="bar",
        legend=False,
        figsize=(7, 4),
    )
    ax.set_title("Contact Class Distribution")
    ax.set_xlabel("Contact Label")
    ax.set_ylabel("Number of Candidate Events")
    plt.tight_layout()
    plt.show()

## 11. Player–Player and Player–Ground Events

In this dataset, the second participant can be another player or the ground.

For a player-ground candidate, `nfl_player_id_2` is represented by `G`.

In [ ]:
# ====================================== Contact Type Audit =========================================

if DATA_AVAILABLE:
    labels_explore = labels.copy()

    labels_explore["contact_type"] = np.where(
        labels_explore["nfl_player_id_2"].astype(str).str.upper().eq("G"),
        "Player-Ground",
        "Player-Player",
    )

    contact_type_summary = (
        labels_explore
        .groupby("contact_type", dropna=False)
        .agg(
            candidate_events=("contact", "size"),
            positive_contacts=("contact", "sum"),
            positive_rate=("contact", "mean"),
        )
        .reset_index()
    )

    contact_type_summary["positive_rate"] *= 100

    display(contact_type_summary)

## 12. Explore One Positive Contact Event

A single labelled positive event is selected deterministically from the dataset.

The purpose is not to draw a conclusion from one event. It is to understand how a contact label connects to:

- `game_play`
- `step`
- player IDs
- player positions in the tracking table.

In [ ]:
# ================================= Select One Positive Event =======================================

if DATA_AVAILABLE:
    positive_events = labels_explore.loc[
        labels_explore["contact"] == 1
    ].copy()

    example_event = positive_events.iloc[0]

    EXAMPLE_GAME_PLAY = example_event["game_play"]
    EXAMPLE_STEP = int(example_event["step"])
    EXAMPLE_PLAYER_1 = example_event["nfl_player_id_1"]
    EXAMPLE_PLAYER_2 = example_event["nfl_player_id_2"]

    print(f"game_play       : {EXAMPLE_GAME_PLAY}")
    print(f"step            : {EXAMPLE_STEP}")
    print(f"nfl_player_id_1 : {EXAMPLE_PLAYER_1}")
    print(f"nfl_player_id_2 : {EXAMPLE_PLAYER_2}")
    print(f"contact         : {example_event['contact']}")

    display(example_event.to_frame(name="value"))

## 13. Tracking Frame at the Contact Step

The next cell plots all tracked players at the selected step.

This provides an intuitive first look at the spatial information that will later be converted into features such as inter-player distance and relative movement.

In [ ]:
# ================================ Plot Tracking at Contact Step ===================================

if DATA_AVAILABLE:
    frame = tracking.loc[
        (tracking["game_play"] == EXAMPLE_GAME_PLAY)
        & (tracking["step"] == EXAMPLE_STEP)
    ].copy()

    print(f"Tracked players at selected step: {len(frame)}")
    display(frame.head())

    fig, ax = plt.subplots(figsize=(11, 5))

    ax.scatter(
        frame["x_position"],
        frame["y_position"],
        s=55,
    )

    for _, row in frame.iterrows():
        ax.annotate(
            str(row["nfl_player_id"]),
            (row["x_position"], row["y_position"]),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set_title(
        f"Player Positions | {EXAMPLE_GAME_PLAY} | Step {EXAMPLE_STEP}"
    )
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

## 14. Movement Around the Contact

A contact is a temporal event, not just a single point.

We therefore inspect a short window around the selected step. This is the first link between the tabular tracking data and the later temporal modelling stage.

In [ ]:
# =============================== Temporal Window Around Contact ==================================

if DATA_AVAILABLE:
    WINDOW_STEPS = 10

    event_window = tracking.loc[
        (tracking["game_play"] == EXAMPLE_GAME_PLAY)
        & (tracking["step"].between(
            EXAMPLE_STEP - WINDOW_STEPS,
            EXAMPLE_STEP + WINDOW_STEPS,
        ))
    ].copy()

    participant_ids = [EXAMPLE_PLAYER_1]

    if str(EXAMPLE_PLAYER_2).upper() != "G":
        try:
            participant_ids.append(int(float(EXAMPLE_PLAYER_2)))
        except (ValueError, TypeError):
            participant_ids.append(EXAMPLE_PLAYER_2)

    participant_window = event_window.loc[
        event_window["nfl_player_id"].isin(participant_ids)
    ].copy()

    print(f"Rows in full event window       : {len(event_window):,}")
    print(f"Rows for labelled participant(s): {len(participant_window):,}")

    display(participant_window.head(20))

In [ ]:
# ================================ Plot Speed Around Contact =======================================

if DATA_AVAILABLE and "speed" in participant_window.columns:
    fig, ax = plt.subplots(figsize=(10, 4))

    for player_id, player_df in participant_window.groupby("nfl_player_id"):
        player_df = player_df.sort_values("step")

        ax.plot(
            player_df["step"],
            player_df["speed"],
            marker="o",
            markersize=3,
            label=f"Player {player_id}",
        )

    ax.axvline(
        EXAMPLE_STEP,
        linestyle="--",
        label="Labelled contact step",
    )

    ax.set_title("Player Speed Around the Contact Event")
    ax.set_xlabel("Tracking Step")
    ax.set_ylabel("Speed")
    ax.legend()
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()
else:
    if DATA_AVAILABLE:
        print("The tracking table does not contain a 'speed' column.")

## 15. Save the Exploration Audit

The notebook saves small summary tables, not the raw Kaggle data.

These outputs make it easier to reproduce the exact data checks used before modelling.

In [ ]:
# ================================= Save Exploration Reports ========================================

if DATA_AVAILABLE:
    tracking_missing.to_csv(
        REPORTS_DIR / "01_tracking_missing_values.csv"
    )

    label_missing.to_csv(
        REPORTS_DIR / "01_label_missing_values.csv"
    )

    class_counts.to_csv(
        REPORTS_DIR / "01_class_distribution.csv",
        index=False,
    )

    contact_type_summary.to_csv(
        REPORTS_DIR / "01_contact_type_summary.csv",
        index=False,
    )

    duplicate_summary.to_csv(
        REPORTS_DIR / "01_duplicate_summary.csv",
        index=False,
    )

    print(f"✅ Exploration reports saved to: {REPORTS_DIR}")

## 16. Notebook 01 Checklist

After this notebook runs successfully, we should understand:

- the raw table structure;
- dataset size;
- missingness;
- duplicate records;
- class imbalance;
- player-player versus player-ground candidates;
- how a labelled contact maps to tracking coordinates and movement around the event.

### Next Step

**Notebook 02: Tracking Feature Engineering**

The next notebook will build a modelling table from the raw tracking and label tables. Initial features will include:

- player-pair distance;
- relative speed;
- acceleration and relative acceleration;
- direction and orientation differences;
- short temporal context around the candidate contact.

No deep model should be trained until these features and the data split have been checked carefully.